# NumGLUE → dataset siap pakai (Colab, end-to-end A1–A3)

Satu notebook untuk seluruh Track A. Input sudah ada di Drive: `data/NumGlue/numglue_{split}_filtered.jsonl` (hasil `numglue_prep.py`).

```
filtered --A1 dedup+subsample--> sample --A2 translate EN→ID--> id
   --A3a generate CoT--> cand --A3b judge--> correct --A3c assemble cara--> final
```

Output akhir: `data/NumGlue/numglue_{split}_final.jsonl` {soal, cara, jawaban, source, cara_source}.

**Runtime Colab:** A2 butuh **GPU** (NLLB). A3 default pakai **Groq API** (tak butuh GPU) — set `GROQ_API_KEY`. Semua langkah **resumable** (re-run = lanjut).

> Tips: mulai dari `SPLITS=['test']` untuk validasi cepat, baru `['dev','train']`.


## 0. Setup

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate sentencepiece sacremoses openai python-dotenv math-verify


In [ ]:
# (Colab) mount Drive tempat repo+data berada.
from google.colab import drive; drive.mount('/content/drive')

import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / 'src' / 'preprop' / 'dedup.py').exists():
            return cand
    raise RuntimeError('repo tak ketemu — set REPO_ROOT manual di bawah')

# Ganti kalau auto-detect gagal (sesuaikan lokasi repo di Drive):
REPO_ROOT = Path('/content/drive/MyDrive/DATA_NLP')
if not (REPO_ROOT / 'src' / 'preprop' / 'dedup.py').exists():
    REPO_ROOT = find_repo_root(Path.cwd())
    for _pth in (REPO_ROOT, REPO_ROOT / 'src'):
        if str(_pth) not in sys.path: sys.path.insert(0, str(_pth))
import os; os.chdir(REPO_ROOT)
print('REPO_ROOT =', REPO_ROOT)


## 1. API key Groq (untuk A3 — generate + judge + fallback)

In [ ]:
import os
try:
    from dotenv import load_dotenv; load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass
if not os.environ.get('GROQ_API_KEY'):
    from getpass import getpass
    os.environ['GROQ_API_KEY'] = getpass('GROQ_API_KEY: ')
print('GROQ key terpasang:', bool(os.environ.get('GROQ_API_KEY')))


## 2. Konfigurasi

In [ ]:
DATA = REPO_ROOT / 'data' / 'NumGlue'
COT  = REPO_ROOT / 'data' / 'cot'
COT.mkdir(parents=True, exist_ok=True)

SPLITS = ['test', 'dev', 'train']   # edit: mulai ['test'] dulu utk validasi
N_MAP  = {'dev': 2000, 'train': 10000, 'test': 3000}
SEED   = 42

N_CAND      = 4          # kandidat CoT per soal (A3a). AIMO pakai banyak; 4 = hemat.
COT_BACKEND = 'api'      # 'api' (Groq, tanpa GPU) atau 'vllm' (GPU lokal)
DECONTAM    = False      # dekontaminasi cross-lingual vs holdout (lemah) — default off

def paths(split):
    return dict(
        filtered = DATA / f'numglue_{split}_filtered.jsonl',
        dedup    = DATA / f'numglue_{split}_dedup.jsonl',
        sample   = DATA / f'numglue_{split}_sample.jsonl',
        idd      = DATA / f'numglue_{split}_id.jsonl',
        cand     = COT  / f'numglue_{split}_cand.jsonl',
        correct  = COT  / f'numglue_{split}_correct.jsonl',
        final    = DATA / f'numglue_{split}_final.jsonl',
    )

for s in SPLITS:
    p = paths(s); print(f'{s:5} filtered ada: {p["filtered"].exists()}')


## A1 — Dedup + Subsample (CPU/GPU ringan)
dedup internal cosine>0.92, lalu stratified-by-`source_type` ke N, lalu **drop `type`**.


In [ ]:
from src.preprop.dedup import run_dedup
from src.preprop import subsample as ss

bench = [str(REPO_ROOT / 'data' / 'eval' / 'holdout_v3_un.jsonl')] if DECONTAM else None
for s in SPLITS:
    p = paths(s)
    print(f'\n== A1 {s} ==')
    print('dedup    :', run_dedup(p['filtered'], p['dedup'], benchmark_paths=bench))
    print('subsample:', ss.run(p['dedup'], p['sample'], n=N_MAP[s], seed=SEED))


## A2 — Translate EN→ID (GPU, NLLB math-safe)
Jawaban numerik di-copy verbatim; hanya teks NL yang diterjemahkan. Resumable per blok.


In [ ]:
from src.scraping_ver2.translate_numglue import run as translate_run
for s in SPLITS:
    p = paths(s)
    print(f'\n== A2 {s} ==')
    print(translate_run(p['sample'], p['idd']))


## A3 — generate `cara` → **pindah ke Kaggle**

A3 (generate + correctness + assemble cara) butuh model lokal (vLLM, tanpa batas token) → jalankan di **`notebooks/numglue_cot_kaggle.ipynb`** pakai input `numglue_{split}_id.jsonl` hasil A2 di atas. Output akhir `numglue_{split}_final.jsonl`.


## 3. Cek hasil akhir

In [ ]:
import json
for s in SPLITS:
    p = paths(s)
    if not p['final'].exists():
        print(f'{s}: belum ada final'); continue
    rows = [json.loads(l) for l in open(p['final'], encoding='utf-8') if l.strip()]
    from collections import Counter
    cs = Counter(r.get('cara_source') for r in rows)
    print(f'{s:5} n={len(rows):5} cara_source={dict(cs)} keys={list(rows[0].keys())}')
print('\nSelesai. File final: data/NumGlue/numglue_{split}_final.jsonl')
